# Holosoma Retargeting Pipeline

## 1. 使用的模型资源类型

- URDF（Unified Robot Description Format）：描述机器人或物体由哪些 link 组成、link 之间用什么 joint 连接。
- MuJoCo XML：优化、正运动学、Jacobian 和碰撞距离查询实际加载的场景文件。
- OBJ/STL/DAE：URDF 或 XML 引用的几何 mesh 资源，`load_object_data(...)` 会从物体 mesh 表面采样点。
- `.pt` / `.npy` / `.npz`：人体动作和物体位姿数据源，具体读取方式由 `task_type` 和 `data_format` 决定。


# 2. `robot_retarget.py` 主流程和数据流

脚本入口是：

```python
if __name__ == "__main__":
    cfg = tyro.cli(RetargetingConfig)
    main(cfg)
```

`tyro.cli(RetargetingConfig)` 会根据 `RetargetingConfig` dataclass 生成命令行接口，并把命令行参数解析成一个 `cfg` 对象。`cfg` 里包含：

- `cfg.robot`：机器人类型，例如 `g1`。
- `cfg.task_type`：任务类型，支持 `robot_only`、`object_interaction`、`climbing`。
- `cfg.data_format`：人体动作格式；如果为 `None`，`main(cfg)` 会根据任务类型选择默认值。
- `cfg.task_name`、`cfg.data_path`、`cfg.save_dir`：输入数据和输出路径相关配置。
- `cfg.robot_config`：机器人 URDF、DOF、高度等配置。
- `cfg.motion_data_config`：人体关节名、人体到机器人 link 的 mapping、toe 名称、默认缩放设置。
- `cfg.task_config`：物体名、地面点网格、climbing 物体目录和采样权重等任务配置。
- `cfg.retargeter`：优化器开关、权重、trust region、foot lock、自碰撞等配置。

不传任何命令行参数时，`cfg` 的默认构造大致等价于：

```python
cfg = RetargetingConfig(
    task_type="object_interaction",
    robot="g1",
    data_format=None,
    task_name="sub3_largebox_003",
    data_path=Path("demo_data/OMOMO_new"),
    save_dir=None,
    augmentation=False,
    robot_config=RobotConfig(robot_type="g1"),
    motion_data_config=MotionDataConfig(data_format="smplh", robot_type="g1"),
    task_config=TaskConfig(),
    retargeter=RetargeterConfig(),
)
```

`main(cfg)` 的职责是把这些配置和数据逐步整理成 `retargeter.retarget_motion(...)` 的输入。


## 2.1 配置校验：`validate_config(cfg)`

调用位置：

```python
validate_config(cfg)
```

输入：

- `cfg: RetargetingConfig`，由 `tyro.cli(...)` 解析得到。

输出 / 返回值：

- 正常情况下没有显式返回值，即返回 `None`。
- 如果配置非法，直接抛出 `ValueError`。

功能：

- 检查 `cfg.data_format` 是否已经注册在 `DEMO_JOINTS_REGISTRY`。
- 检查任务和数据格式是否匹配：`climbing` 只接受 `mocap` 或默认 `None`，`object_interaction` 只接受 `smplh` 或默认 `None`。
- `robot_only` 接受 registry 中的任意数据格式。

这个函数只做配置一致性检查，不读取数据、不创建模型、不修改运动轨迹。


## 2.2 选择默认配置并同步嵌套 config

`main(cfg)` 开始后先取出：

```python
robot = cfg.robot
task_name = cfg.task_name
task_type = cfg.task_type
```

然后设置默认数据格式和输出目录：

```python
data_format = cfg.data_format or DEFAULT_DATA_FORMATS[task_type]
save_dir = cfg.save_dir if cfg.save_dir is not None else Path(DEFAULT_SAVE_DIRS[task_type].format(robot=robot))
data_path = cfg.data_path
```

默认 `data_format` 来自：

```python
DEFAULT_DATA_FORMATS = {
    "robot_only": "smplh",
    "object_interaction": "smplh",
    "climbing": "mocap",
}
```

接着同步嵌套配置：

```python
if cfg.robot_config.robot_type != robot:
    cfg.robot_config = RobotConfig(robot_type=robot)

if cfg.motion_data_config.robot_type != robot or cfg.motion_data_config.data_format != data_format:
    cfg.motion_data_config = MotionDataConfig(data_format=data_format, robot_type=robot)
```

这一步的意义是：顶层 `cfg.robot` / `data_format` 是最终选择，嵌套的 `robot_config` 和 `motion_data_config` 必须和它们一致，否则后面会查到错误的机器人 DOF、身高、`JOINTS_MAPPING` 或 `DEMO_JOINTS`。

如果是 `climbing` 且没有传 `cfg.task_config.object_dir`，代码会默认设置为：

```python
object_dir = data_path / task_name
```


## 2.3 任务常量合并：`create_task_constants(...)`

调用位置：

```python
constants = create_task_constants(
    robot_config=cfg.robot_config,
    motion_data_config=cfg.motion_data_config,
    task_config=cfg.task_config,
    task_type=task_type,
)
```

输入：

- `robot_config`：机器人配置，包含 `ROBOT_URDF_FILE`、`ROBOT_DOF`、`ROBOT_HEIGHT` 等大写常量。
- `motion_data_config`：动作数据格式配置，负责 `DEMO_JOINTS`、`JOINTS_MAPPING`、`TOE_NAMES`、默认身高或缩放参数。
- `task_config`：任务配置，负责 `object_name`、ground 网格、climbing object directory 等。
- `task_type`：决定物体相关常量如何设置。

输出 / 返回值：

- `task_constants: SimpleNamespace`。

主要功能：

1. 把 `robot_config` 中的大写属性复制到 `task_constants`。
2. 调用 `motion_data_config.legacy_constants()`，把动作数据相关常量复制进去。
3. 根据 `task_type` 设置 `OBJECT_NAME`、`OBJECT_URDF_FILE`、`OBJECT_MESH_FILE`、`SCENE_XML_FILE` 等物体/场景路径。

`JOINTS_MAPPING` 的来源尤其重要。代码不是在 `robot_retarget.py` 里直接写 mapping，而是间接调用：

```python
motion_data_config.legacy_constants()
```

其中包含：

```python
"JOINTS_MAPPING": self.resolved_joints_mapping
```

`resolved_joints_mapping` 会用：

```python
key = (self.data_format, self.robot_type)
return JOINTS_MAPPINGS[key]
```

例如默认 `object_interaction + g1` 时，`data_format="smplh"`、`robot_type="g1"`，因此查的是：

```python
JOINTS_MAPPINGS[("smplh", "g1")]
```

这个字典表示：

```text
demo / human joint name -> robot MuJoCo body / link name
```

后面 `InteractionMeshRetargeter.__init__` 会把它保存为：

```python
self.laplacian_match_links = task_constants.JOINTS_MAPPING
```


## 2.4 数据导入：`load_motion_data(...)`

调用位置：

```python
human_joints, object_poses, smpl_scale = load_motion_data(
    task_type, data_format, data_path, task_name, constants, cfg.motion_data_config
)
```

输入：

- `task_type`：决定走 `robot_only`、`object_interaction` 还是 `climbing` 的读取逻辑。
- `data_format`：决定读取 `.pt`、`.npy`、`.npz`，以及人体关节名解释方式。
- `data_path`：输入数据根目录。
- `task_name`：动作序列名，用于拼接文件路径。
- `constants`：当前任务常量，主要用到 `ROBOT_HEIGHT`、`DEMO_JOINTS` 等。
- `motion_data_config`：数据格式配置，提供默认身高、默认缩放等。

输出 / 返回值：

```python
human_joints, object_poses, smpl_scale
```

- `human_joints`：人体关节世界坐标轨迹，形状 `(T, J, 3)`。
- `object_poses`：物体位姿轨迹，形状 `(T, 7)`，刚读出时顺序是 `[qw, qx, qy, qz, x, y, z]`。
- `smpl_scale`：把人体示范尺度调整到机器人尺度附近的系数，常见形式是 `robot_height / human_height`。

分支行为：

- `robot_only + lafan`：读取 `.npy`，做 y-up 到 z-up 坐标转换，使用默认缩放 `1.27 / 1.7`。
- `robot_only + smplh`：读取 InterMimic `.pt`，从序列名对应的人体身高计算 `smpl_scale`。
- `robot_only + mocap`：读取 `.npy` 并降采样，默认人体身高 `1.78m`。
- `robot_only + smplx` 或类似 `.npz`：读取 `global_joint_positions` 和 `height`。
- `object_interaction`：读取 InterMimic `.pt`，返回真实 `human_joints` 和 `object_poses`，并计算 `smpl_scale`。
- `climbing`：读取任务目录下 `.npy`，构造 dummy object pose，默认人体身高 `1.78m`。

`robot_only` 和 `climbing` 没有真实 moving object pose 时，会构造：

```python
object_poses[t] = [1, 0, 0, 0, 0, 0, 0]
```

这里 `[1,0,0,0]` 是单位四元数，`[0,0,0]` 是物体位置。


## 2.5 物体和场景几何准备：`setup_object_data(...)`

调用位置：

```python
object_local_pts, object_local_pts_demo, object_urdf_path = setup_object_data(
    task_type,
    constants,
    cfg.task_config.object_dir,
    smpl_scale,
    cfg.task_config,
    cfg.augmentation,
    object_scale_augmented=_OBJECT_SCALE_AUGMENTED,
)
```

输入：

- `task_type`：决定准备 ground points、普通物体点还是 climbing terrain。
- `constants`：包含 `OBJECT_MESH_FILE`、`OBJECT_URDF_FILE`、`SCENE_XML_FILE` 等。
- `object_dir`：climbing 任务的物体目录。
- `smpl_scale`：来自 `load_motion_data(...)` 的缩放系数。
- `task_config`：ground 网格范围、采样权重、object name 等配置。
- `augmentation`：是否生成增强物体几何。
- `object_scale_augmented`：climbing augmentation 里的物体额外缩放，默认 `[1.0, 1.0, 1.2]`。

输出 / 返回值：

```python
object_local_pts, object_local_pts_demo, object_urdf_path
```

- `object_local_pts`：current/robot 侧使用的物体点或地面点。
- `object_local_pts_demo`：demo/source 侧用来构造目标 interaction mesh 的物体点或地面点。
- `object_urdf_path`：传给 retargeter 加载物体模型的 URDF 路径；`robot_only` 为 `None`。

两个 obj pts 的区别：

- `object_local_pts` 后面会作为 `object_points_local` 传入 `retarget_motion(...)`。它属于 current/robot 侧，后续在 `solve_single_iteration(...)` 里作为 `obj_pts_local`，和机器人 keypoints 拼成当前 interaction mesh。
- `object_local_pts_demo` 后面会作为 `object_points_local_demo` 传入 `retarget_motion(...)`。它属于 demo/source 侧，后续和人体关节一起构造 `source_vertices`，并生成目标 Laplacian。

分支行为：

- `robot_only`：调用 `create_ground_points(...)` 创建地面网格点，返回 `ground_pts, ground_pts, None`。这时 `object_local_pts` 和 `object_local_pts_demo` 完全相同；代码里没有真实物体，地面点既作为 source 侧环境点，也作为 current 侧环境点。
- `object_interaction`：调用 `load_object_data(constants.OBJECT_MESH_FILE, smpl_scale=smpl_scale, sample_count=100)` 从物体表面采样点，返回 `points, points * smpl_scale, constants.OBJECT_URDF_FILE`。这时 `object_local_pts = points`，current/robot 侧使用原始物体采样点；`object_local_pts_demo = points * smpl_scale`，demo/source 侧使用按人体缩放系数缩放后的物体采样点。
- `climbing`，暂时不纳入此说明范围。

`object_interaction`有一个需要注意的尺度问题: `load_object_data(...)` 返回顺序是：

```python
return points, points_scaled
```

所以当前代码里：

```text
object_local_pts      = points               # 原始物体点
object_local_pts_demo = points * smpl_scale  # 缩放后的 demo/source 侧物体点
```

**后面 demo/source 侧会使用“缩放后的人 + 缩放后的物体点”，current/robot 侧会使用“机器人 + 原始物体点”。这里似乎有点问题。**

## 2.6 Retargeter 创建

调用位置：

```python
retargeter_kwargs = build_retargeter_kwargs_from_config(
    cfg.retargeter, constants, object_urdf_path, task_type
)
retargeter = InteractionMeshRetargeter(**retargeter_kwargs)
```

`build_retargeter_kwargs_from_config(...)` 输入：

- `retargeter_config`：来自 `cfg.retargeter`，包含优化开关、权重、step size、foot lock/self collision 配置。
- `constants`：上一节创建的任务常量。
- `object_urdf_path`：来自 `setup_object_data(...)`。
- `task_type`：用于 climbing 时额外传入 `nominal_tracking_tau`。

输出：

- `kwargs: dict`，直接作为 `InteractionMeshRetargeter(...)` 的初始化参数。

`InteractionMeshRetargeter(...)` 初始化后会保存：

- MuJoCo model/data：用于 FK、Jacobian、碰撞查询。
- `self.object_name`：来自 `task_constants.OBJECT_NAME`，决定是否使用物体坐标系。
- `self.demo_joints`：当前人体数据格式的关节名表。
- `self.laplacian_match_links`：人体 joint 到机器人 link 的 mapping。
- `self.smplh_mapped_joint_indices`：参与 Laplacian matching 的人体关节索引。
- `self.q_a_indices`：当前优化允许修改的 qpos 维度。
- 约束开关和权重：joint limits、object non-penetration、foot sticking、self collision、`laplacian_weights`、`smooth_weight`、`step_size` 等。

这里不展开 `visualize` 和 `debug` 分支。


## 2.7 动作预处理：`preprocess_motion_data(...)`

调用位置：

```python
if task_type == "robot_only":
    human_joints = preprocess_motion_data(human_joints, retargeter, toe_names, smpl_scale)
elif task_type in {"object_interaction", "climbing"}:
    human_joints, object_poses, object_moving_frame_idx = preprocess_motion_data(
        human_joints,
        retargeter,
        toe_names,
        scale=smpl_scale,
        object_poses=object_poses,
    )
```

输入：

- `human_joints`：`load_motion_data(...)` 返回的人体关节轨迹。
- `retargeter`：提供 `retargeter.demo_joints`，用于查 toe 的关节索引。
- `toe_names`：当前数据格式下左右脚趾名称。
- `scale` / `smpl_scale`：人体到机器人尺度的缩放系数。
- `object_poses`：可选，只有 object interaction / climbing 分支会传入。

输出：

- `robot_only`：只返回预处理后的 `human_joints`。
- `object_interaction` / `climbing`：返回 `human_joints`、`object_poses`、`object_moving_frame_idx`。

功能：

1. 用左右 toe 的最低高度估计地面高度，把整段人体动作沿 z 方向平移到地面附近。
2. `human_joints = human_joints * scale`，把人体骨架缩放到机器人尺度附近。
3. 如果传入 `object_poses`，缩放物体 pose 的平移部分：`x/y` 直接乘以 `scale`，`z` 保持第一帧高度不变，只缩放相对第一帧的高度变化量。
4. 提取物体第一次明显开始运动的帧 `object_moving_frame_idx`。

这一步会覆盖 `main(cfg)` 里的 `human_joints` 和 `object_poses` 变量。后面所有 demo target 都基于预处理后的数据。


## 2.8 机器人初值和物体位姿顺序：`initialize_robot_pose(...)`

调用位置：

```python
q_init, q_nominal, object_poses_augmented, human_joints, object_poses = initialize_robot_pose(
    task_type,
    data_format,
    human_joints,
    object_poses,
    constants,
    retargeter,
    cfg.task_config,
    cfg.augmentation,
    save_dir,
    task_name,
    augmentation_translation=_AUGMENTATION_TRANSLATION,
)
```

输入：

- `task_type` / `data_format`：决定初始化根节点位置时用 root 还是 `Spine1`。
- `human_joints`：预处理后的 demo 人体轨迹。
- `object_poses`：预处理后的 demo 物体轨迹，此时仍是 `[qw, qx, qy, qz, x, y, z]` 顺序。
- `constants`：提供 `ROBOT_DOF`、`DEMO_JOINTS` 等。
- `retargeter`：climbing 分支需要用它查 `Spine1` 的索引。
- `augmentation`、`save_dir`、`task_name`：决定是否读取原始 retarget 结果作为 nominal trajectory。

输出 / 返回值：

```python
q_init, q_nominal, object_poses_augmented, human_joints, object_poses
```

- `q_init`：机器人初始 qpos，格式是 `[base_x, base_y, base_z, base_qw, base_qx, base_qy, base_qz, joint_1, ...]`。物体 pose 不在这个数组里。
- `q_nominal`：augmentation 模式下读取的原始 retarget 轨迹；非 augmentation 时为 `None`。
- `object_poses_augmented`：求解时实际锁定到 qpos 最后 7 维的物体轨迹。非 augmentation 时等于原始 `object_poses` 的 copy。
- `human_joints`：在这里是原样返回。
- `object_poses`：demo/source 侧物体轨迹，转换成 MuJoCo 顺序 `[x, y, z, qw, qx, qy, qz]`。

`q_init` 的计算由 `_compute_q_init_base(...)` 完成：

- 位置：通常取第 0 帧人体 root 或 `Spine1` 的 xyz。
- 朝向：用“人到物体”的水平向量定义初始 x 轴，z 轴向上，再转成四元数。
- 关节：`np.zeros(constants.ROBOT_DOF)`，即所有机器人关节初始为 0。

物体 pose 顺序转换由：

```python
convert_object_poses_to_mujoco_order(object_poses)
```

完成：

```python
[qw, qx, qy, qz, x, y, z] -> [x, y, z, qw, qx, qy, qz]
```

后面 `retarget_motion(...)` 中会按 MuJoCo 顺序使用：

```python
object_trans_demo = object_poses[i, :3]
object_quat_demo = object_poses[i, 3:]
```


## 2.9 脚接触序列：`extract_foot_sticking_sequence_velocity(...)`

调用位置：

```python
foot_sticking_sequences = extract_foot_sticking_sequence_velocity(
    human_joints, retargeter.demo_joints, toe_names
)
```

输入：

- `human_joints`：预处理后的源人体关节轨迹，形状 `(T, J, 3)`。
- `retargeter.demo_joints`：人体关节名列表，用于把 toe 名称转成关节 index。
- `toe_names`：左右脚趾名称，例如 `['L_Toe', 'R_Toe']`。
- `velocity_threshold`：默认 `0.01`。代码实际比较的是相邻帧 xy 位移长度，没有除以帧间隔。

输出：

```python
foot_sticking_sequences == [
    {"L_Toe": False, "R_Toe": False},
    {"L_Toe": True,  "R_Toe": False},
    ...
]
```

功能：

- 取左右 toe 的水平坐标 `:2`。
- 计算相邻帧水平位移长度。
- 位移小于阈值时，该脚在该帧标记为 sticking。
- 第 0 帧默认不 sticking。

如果 `task_type == "object_interaction"`，代码会额外把第 0 帧左右脚都设为 `False`。后续 `solve_single_iteration(...)` 会根据这些布尔值给对应机器人脚 link 加 XY 防滑约束。


## 2.10 输出路径和核心优化入口

输出路径由：

```python
dest_res_path = determine_output_path(task_type, save_dir, task_name, cfg.augmentation)
```

生成：

- `robot_only`：`{save_dir}/{task_name}.npz`
- `object_interaction` / `climbing`：`{save_dir}/{task_name}_original.npz` 或 `{task_name}_augmented.npz`

最后调用：

```python
retargeter.retarget_motion(
    human_joint_motions=human_joints,
    object_poses=object_poses,
    object_poses_augmented=object_poses_augmented,
    object_points_local_demo=object_local_pts_demo,
    object_points_local=object_local_pts,
    foot_sticking_sequences=foot_sticking_sequences,
    q_a_init=q_init,
    q_nominal_list=q_nominal,
    original=not cfg.augmentation,
    dest_res_path=dest_res_path,
)
```

这些参数的来源和用途：

- `human_joint_motions` 来自预处理后的 `human_joints`。retargeter 每帧从里面取人体 keypoints，用来构造 demo/source interaction mesh。
- `object_poses` 来自 `initialize_robot_pose(...)` 转换后的 demo 物体 pose。它用于把 demo 人体点转到 demo 物体坐标系，并生成 target Laplacian。
- `object_poses_augmented` 也来自 `initialize_robot_pose(...)`。它会写入 `q_locked_list[:, -7:]`，作为求解时锁定的 target/current 物体 pose。
- `object_points_local_demo` 来自 `setup_object_data(...)` 的 `object_local_pts_demo`。它是 demo/source 侧物体点，参与 `source_vertices` 和 `target_laplacian`。
- `object_points_local` 来自 `setup_object_data(...)` 的 `object_local_pts`。它是 current/robot 侧物体点，后面和机器人点组成当前 mesh。
- `foot_sticking_sequences` 来自 `extract_foot_sticking_sequence_velocity(...)`，决定每帧哪些脚加 XY 防滑约束。
- `q_a_init` 来自 `initialize_robot_pose(...)` 的 `q_init`。没有 nominal trajectory 时，它用于第 0 帧机器人初值。
- `q_nominal_list` 在 augmentation 时来自原始 run 保存的 qpos 轨迹，用作锁定模板和 nominal tracking 参考；非 augmentation 时为 `None`。
- `original` 是 `not cfg.augmentation`，决定 nominal tracking 权重是否随帧数衰减。
- `dest_res_path` 来自 `determine_output_path(...)`，用于保存 `.npz` 结果。


# 3. `InteractionMeshRetargeter` 的内部主路径

`retargeter.retarget_motion(...)` 实现在 `interaction_mesh_retargeter_个人注释版.py` 的 `InteractionMeshRetargeter` 类中。核心流程是：外层逐帧构造 demo/source 的 Laplacian target，内层用 SQP-style 迭代求每帧机器人 qpos。

主调用链：

```text
InteractionMeshRetargeter.__init__(...)
  -> retarget_motion(...)
      初始化 q_locked_list / q / retargeted_motions
      for each frame i:
        构造 demo/source interaction mesh
        计算 target_laplacian
        -> iterate(...)
            repeat:
              -> solve_single_iteration(...)
                  初始化当前线性化点 q
                  计算机器人 keypoint 位置和 Jacobian
                  构造局部凸子问题
                  求解 dqa 并更新 q
        保存该帧 q
  -> 保存 .npz
```

下面只讲优化主路径和关键辅助函数，跳过 `debug` 可视化细节。


## 3.1 初始化层：`__init__(...)`

`__init__(...)` 的输入来自 `build_retargeter_kwargs_from_config(...)`，主要包括：

- `task_constants`：含机器人、人体关节、mapping、物体路径等常量。
- `object_urdf_path`：物体 URDF 路径，`robot_only` 时是 `None`。
- `q_a_init_idx`：决定从 qpos 的哪一维开始优化。
- 各类约束开关：joint limits、object non-penetration、foot sticking、self collision。
- 优化参数：`step_size`、`penetration_tolerance`、`foot_sticking_tolerance`、`w_nominal_tracking_init`、`nominal_tracking_tau`。

核心初始化结果：

- `self.object_name = task_constants.OBJECT_NAME`。默认 `robot_only` 是 `ground`，所以后续 `self.object_name != "ground"` 为 `False`，Laplacian keypoints 直接在世界系里算。
- `self.robot_model` / `self.robot_data`。MuJoCo 的 model/data，后续 `qpos`、`xpos`、Jacobian、碰撞查询都依赖它。
- `self.has_dynamic_object`。如果 MuJoCo `qpos` 长度大于 `7 + ROBOT_DOF`，说明场景里还有动态物体 free joint。
- `self.nq = self.robot_model.nq`。完整 qpos 维度。
- `self.q_a_indices = np.arange(7 + self.q_a_init_idx, 7 + ROBOT_DOF)`。当前优化变量在 full qpos 里的列索引。
- `self.nq_a = len(self.q_a_indices)`。局部优化变量数量。
- `self.demo_joints = task_constants.DEMO_JOINTS`。人体关节名顺序。
- `self.laplacian_match_links = task_constants.JOINTS_MAPPING`。人体 joint 名到机器人 body/link 名的 mapping。
- `self.smplh_mapped_joint_indices`。把 mapping 的 key 从 joint 名转换成 `human_joint_motions` 的整数索引。
- `self.q_a_lb / self.q_a_ub`。优化变量的上下界，由 MuJoCo joint range 和手工 override 合成。

这些初始化值会贯穿 `retarget_motion(...)`、`solve_single_iteration(...)` 和 Jacobian 计算。


## 3.2 外层入口：`retarget_motion(...)` 的输入输出

函数签名：

```python
def retarget_motion(
    self,
    human_joint_motions,
    object_poses,
    object_poses_augmented,
    object_points_local_demo,
    object_points_local,
    foot_sticking_sequences,
    q_a_init=None,
    q_nominal_list=None,
    original=True,
    dest_res_path=None,
):
```

输入来源：

- `human_joint_motions`：来自 `main(cfg)` 中预处理后的 `human_joints`。
- `object_poses`：来自 `initialize_robot_pose(...)`，是 MuJoCo 顺序的 demo/source 物体 pose。
- `object_poses_augmented`：来自 `initialize_robot_pose(...)`，是 current/target 侧要锁定到 qpos 里的物体 pose。
- `object_points_local_demo`：来自 `setup_object_data(...)`，是 demo/source 侧物体点。
- `object_points_local`：来自 `setup_object_data(...)`，是 current/robot 侧物体点。
- `foot_sticking_sequences`：来自 `extract_foot_sticking_sequence_velocity(...)`。
- `q_a_init`：来自 `initialize_robot_pose(...)` 的 `q_init`。
- `q_nominal_list`：augmentation 时来自原始 retarget 结果；非 augmentation 时为 `None`。
- `original`：来自 `not cfg.augmentation`。
- `dest_res_path`：来自 `determine_output_path(...)`。

返回值：

```python
retargeted_motions, obj_pts_demo_list, obj_pts_list, tetrahedra
```

- `retargeted_motions`：优化后的 qpos 序列，返回时去掉初始化占位的第 0 个元素。
- `obj_pts_demo_list` / `obj_pts_list`：debug 可视化时保存的物体点世界坐标；非 debug 下通常为空。
- `tetrahedra`：每帧 source interaction mesh 的 Delaunay 四面体拓扑。

函数还会保存 `.npz`：

```python
np.savez(dest_res_path, qpos=..., human_joints=..., fps=30, cost=cost)
```


## 3.3 `retarget_motion(...)` 循环前初始化

循环前代码：

```python
num_frames = human_joint_motions.shape[0]
if q_nominal_list is not None:
    q_locked_list = q_nominal_list
else:
    q_locked_list = np.zeros((num_frames, self.nq))
    q_locked_list[0, self.q_a_indices] = q_a_init

q_locked_list[:, -7:] = object_poses_augmented
q = np.copy(q_locked_list[0])
retargeted_motions = [q]

tetrahedra = []
obj_pts_demo_list = []
obj_pts_list = []
```

逐个变量解释：

- `num_frames`：总帧数，来自 `human_joint_motions.shape[0]`。而 `human_joint_motions` 来自 `main(cfg)` 里预处理后的 `human_joints`。
- `q_locked_list`：每一帧的 full qpos 背景模板。
- `q_nominal_list is not None`：augmentation 模式下，`q_nominal_list` 来自原始 run 保存的 `.npz["qpos"]`，所以直接用它作为每帧背景和 nominal tracking 参考。
- `q_nominal_list is None`：普通模式下创建全 0 的 `(num_frames, self.nq)`，只把第 0 帧的 active variables 写成 `q_a_init`。
- `q_locked_list[0, self.q_a_indices] = q_a_init`：第 0 帧机器人优化变量来自 `initialize_robot_pose(...)` 算出的 `q_init`，即 `[base position, base quaternion, zero joints]` 的机器人初始姿态。
- `q_locked_list[:, -7:] = object_poses_augmented`：把每一帧 qpos 最后 7 维设为 current/target 侧物体 pose。这个变量来自 `initialize_robot_pose(...)`；非 augmentation 时等于 demo 物体 pose，augmentation 时是扰动后的目标物体 pose。
- `q = np.copy(q_locked_list[0])`：当前迭代的 q 初值来自第 0 帧锁定模板。
- `retargeted_motions = [q]`：先把初值放进去，后面每帧求解后 append 新 q。循环中 `q_t_last=retargeted_motions[-1]` 会用它作为上一帧参考，参与 smoothness 和 foot sticking。
- `tetrahedra`：保存每帧 source mesh 的四面体拓扑。
- `obj_pts_demo_list` / `obj_pts_list`：仅用于 debug 可视化数据保存，不是优化主变量。

`q_locked_list` 的名字里 “locked” 的意思是：它保存当前帧不希望优化器随便改的背景量，尤其是物体 pose。真正局部优化时会复制 `q_locked_list[i]`，再把 `self.q_a_indices` 对应的机器人变量覆盖成当前迭代值。


## 3.4 每帧循环内变量的数据来源

每一帧 `i` 的主流程从 demo/source 侧数据开始：

```python
object_quat_demo = object_poses[i, 3:]
object_trans_demo = object_poses[i, :3]
human_mapped_joints = human_joint_motions[i, self.smplh_mapped_joint_indices]
```

数据来源：

- `object_poses[i]`：来自 `initialize_robot_pose(...)` 转换后的 demo/source 物体 pose，顺序是 `[x, y, z, qw, qx, qy, qz]`。
- `object_trans_demo`：demo 物体第 `i` 帧的位置 `[x, y, z]`。
- `object_quat_demo`：demo 物体第 `i` 帧的四元数 `[qw, qx, qy, qz]`。
- `human_joint_motions[i]`：预处理后的 demo 人体第 `i` 帧所有关节。
- `self.smplh_mapped_joint_indices`：由 `DEMO_JOINTS` 和 `JOINTS_MAPPING` 算出，只取参与 Laplacian matching 的人体 keypoints。

然后处理坐标系：

```python
if self.object_name == "ground":
    human_mapped_joints_in_object = human_mapped_joints
else:
    human_mapped_joints_in_object = transform_points_world_to_local(
        object_quat_demo, object_trans_demo, human_mapped_joints
    )
```

- `ground` / `robot_only`：没有真实交互物体，人体 keypoints 保持在世界系。
- 普通 object interaction：把人体 keypoints 从世界系转到 demo 物体局部坐标系。原因是 `object_points_local_demo` 本来就在物体局部坐标系下。

构造 demo/source interaction mesh：

```python
source_vertices, source_tetrahedra = create_interaction_mesh(
    np.vstack([human_mapped_joints_in_object, object_points_local_demo])
)
```

这里：

- 前半部分是 demo/source 侧人体 keypoints。
- 后半部分是 demo/source 侧物体点 `object_points_local_demo`。
- `create_interaction_mesh(...)` 用 Delaunay 把这些点连成四面体 mesh。

然后生成优化目标：

```python
adj_list = get_adjacency_list(source_tetrahedra, len(source_vertices))
target_laplacian = calculate_laplacian_coordinates(source_vertices, adj_list)
```

- `adj_list`：由四面体拓扑得到的图邻接表。
- `target_laplacian`：demo/source mesh 的 Laplacian coordinates，是这一帧机器人要匹配的目标。

调用内层迭代：

```python
q, cost = self.iterate(
    q_locked=q_locked_list[i],
    q_n=q,
    q_t_last=retargeted_motions[-1],
    target_laplacian=target_laplacian,
    adj_list=adj_list,
    obj_pts_local=object_points_local,
    foot_sticking=foot_sticking_sequences[i],
    ...
)
```

关键数据来源：

- `q_locked_list[i]`：当前帧锁定模板，含当前帧 target/current 物体 pose。
- `q_n=q`：上一轮/上一帧传下来的当前机器人状态。
- `q_t_last=retargeted_motions[-1]`：上一帧最终解，用于 smoothness 和 foot sticking。
- `target_laplacian`：当前帧 demo/source 目标。
- `adj_list`：当前帧 source mesh 拓扑转换成的邻接关系，后面 current/robot 侧也复用同一套邻接关系。
- `obj_pts_local=object_points_local`：current/robot 侧物体点，将和机器人 keypoints 拼成当前 mesh。
- `foot_sticking_sequences[i]`：第 `i` 帧左右脚 sticking 判断。

循环末尾：

```python
retargeted_motions.append(q)
```

把当前帧优化结果保存起来，供下一帧作为 temporal reference。


## 3.5 demo interaction mesh 和 `target_laplacian`

`source_vertices` 是 demo/source 侧的点集。记参与匹配的人体点数量为 $V_r$，物体点数量为 $V_o$，则：

$$
V_s(i)=
\begin{bmatrix}
h_1(i)\\
\vdots\\
h_{V_r}(i)\\
o_1\\
\vdots\\
o_{V_o}
\end{bmatrix}
\in \mathbb{R}^{(V_r+V_o)\times 3}.
$$

在 object interaction 中，这些点都表达在 demo 物体局部坐标系下；在 `ground` 任务中，它们表达在世界系下。

`create_interaction_mesh(...)` 对 `source_vertices` 做 Delaunay 四面体剖分：

```python
tri = Delaunay(vertices)
return vertices, tri.simplices
```

`source_tetrahedra` 是四面体顶点索引表。`get_adjacency_list(...)` 把每个四面体内部的 4 个顶点两两连接，得到每个 vertex 的邻居集合。

`calculate_laplacian_coordinates(...)` 对每个顶点计算：

$$
\delta_l = v_l - \frac{1}{|\mathcal{N}(l)|}
\sum_{s\in\mathcal{N}(l)}v_s.
$$

最终：

```python
target_laplacian.shape == source_vertices.shape == (V_r + V_o, 3)
```



### 3.5.1 论文中使用 object frame 的理论依据

OmniRetarget 论文在 **Interaction Mesh Construction in Object Frame** 小节明确说明：对于 robot-object interaction，interaction mesh 应该构造在物体局部坐标系中。论文给出的理由是：Laplacian coordinates 编码的是相对空间关系；如果在物体坐标系中计算，它们对物体的全局平移和全局旋转保持不变，因此更适合保持人与物体之间的交互几何关系。


论文原文的核心意思可以概括为：object-frame Laplacian coordinates 对物体的 global rotation 和 translation 不变；这对保持 intended interaction geometry 是必要的。当前源码也在类 docstring 中写明优化代价是 “Minimize the Laplacian deformation in the object frame”，并在每帧构造 source mesh 时用 `transform_points_world_to_local(...)` 实现这个选择。


## 3.6 每帧内层迭代：`iterate(...)`

函数签名：

```python
def iterate(
    self,
    q_locked,
    q_n,
    q_t_last,
    target_laplacian,
    adj_list,
    obj_pts_local,
    foot_sticking,
    w_nominal_tracking=0.0,
    q_a_nominal=None,
    init_t=False,
    n_iter=10,
    frame_idx=0,
):
```

输入：

- `q_locked`：当前帧锁定模板，来自 `q_locked_list[i]`。
- `q_n`：当前帧当前迭代点，第一次进入时通常是上一帧解或第 0 帧初值。
- `q_t_last`：上一帧最终 q，用于 smoothness 和 foot sticking。
- `target_laplacian`：当前帧 demo/source Laplacian 目标。
- `adj_list`：当前帧 interaction mesh 邻接表。
- `obj_pts_local`：current/robot 侧物体点。
- `foot_sticking`：当前帧左右脚 sticking 字典。
- `q_a_nominal`：augmentation 模式下当前帧 nominal active variables。
- `init_t`：是否第 0 帧。
- `n_iter`：当前帧最多局部迭代次数，第 0 帧通常 50，后续帧通常 10。
- `frame_idx`：帧编号，用于 foot lock window、自碰撞等按帧配置。

输出：

```python
q_n, cost
```

- `q_n`：当前帧最终 q。
- `cost`：最后一次局部凸子问题的目标值。

循环逻辑：

```python
last_cost = np.inf
for _ in range(n_iter):
    q_a_n_last = q_n[self.q_a_indices]
    q_n, cost = self.solve_single_iteration(...)
    if np.isclose(cost, last_cost):
        break
    last_cost = cost
return q_n, cost
```

每次迭代都会从当前 `q_n` 中取 active variables：

```python
q_a_n_last = q_n[self.q_a_indices]
```

然后调用 `solve_single_iteration(...)` 重新线性化并求一个局部凸子问题。如果 cost 和上一轮接近，就提前停止。


## 3.7 单次局部凸子问题：`solve_single_iteration(...)` 初始化

函数开头的核心初始化：

```python
assert len(q_a_n_last) == self.nq_a
q = np.copy(q_locked)
q[self.q_a_indices] = q_a_n_last
```

数据来源和含义：

- `q_locked`：来自 `iterate(...)` 的当前帧 `q_locked_list[i]`，里面已经写入当前帧 target/current 物体 pose。
- `q_a_n_last`：来自当前迭代点 `q_n[self.q_a_indices]`，表示当前帧上一轮迭代后的机器人 active variables。
- `q = np.copy(q_locked)`：先复制当前帧锁定模板，确保未优化维度和物体 pose 有正确背景。
- `q[self.q_a_indices] = q_a_n_last`：把机器人当前迭代值写回 full qpos，得到本次线性化点。

接着计算机器人 keypoints 的位置和 Jacobian：

```python
J_OC_dict, p_OC_dict, _ = self._calc_manipulator_jacobians(
    q, links=self.laplacian_match_links, obj_frame=(self.object_name != "ground")
)
```

- 如果 `self.object_name == "ground"`，返回的是世界系下的位置/Jacobian。
- 如果有真实物体，返回的是物体坐标系下的位置/Jacobian。
- `p_OC_dict[name]`：机器人某个匹配 link 的当前点位置。
- `J_OC_dict[name]`：该点位置对 active variables 的 Jacobian。

然后初始化 current/robot 侧 mesh：

```python
robot_link_keys = list(self.laplacian_match_links.keys())
V_r = len(robot_link_keys)
V_o = len(obj_pts_local)
V = V_r + V_o

J_V = np.zeros((3 * V, self.nq_a))
for i, key in enumerate(robot_link_keys):
    J_V[3 * i : 3 * (i + 1), :] = J_OC_dict[key]

robot_pts_local = np.array([p_OC_dict[k] for k in robot_link_keys])
vertices = np.vstack([robot_pts_local, obj_pts_local])
```

这里：

- `robot_pts_local`：current/robot 侧机器人 keypoints。
- `obj_pts_local`：current/robot 侧物体点，来自 `retarget_motion(...)` 的 `object_points_local`。
- `vertices`：current/robot 侧 interaction mesh 点集，用来计算当前 Laplacian。
- `J_V`：所有 vertices 对 active variables 的位置 Jacobian。物体点被锁定，所以它们对应的 Jacobian 行保持 0。

这一步完成后，局部问题有了当前线性化点、当前 mesh、当前 mesh 的 Jacobian。


## 3.8 Laplacian 线性化和局部变量

上一层在 demo/source 侧先完成两件事：

```python
source_vertices, source_tetrahedra = create_interaction_mesh(...)
adj_list = get_adjacency_list(source_tetrahedra, len(source_vertices))
target_laplacian = calculate_laplacian_coordinates(source_vertices, adj_list)
```

`adj_list` 和 `target_laplacian` 一起传进 `solve_single_iteration(...)`：demo/source 侧确定 interaction mesh 的拓扑和目标 Laplacian；current/robot 侧按同样的顶点顺序组装当前坐标，然后在这套固定拓扑上计算当前 Laplacian。

current/robot 侧当前坐标是：

```python
vertices = np.vstack([robot_pts_local, obj_pts_local])
```

前 `V_r` 个是人体 keypoints / 机器人 keypoints 的对应点，后 `V_o` 个是 demo 物体点 / current 物体点的对应点。

代码用这个共享 `adj_list` 在 current/robot 侧构造当前 Laplacian：

```python
L = calculate_laplacian_matrix(vertices, adj_list)
lap0 = L @ vertices
lap0_vec = lap0.reshape(-1)
target_lap_vec = target_laplacian.reshape(-1)
```

这里 `lap0_vec` 是当前线性化点的 current/robot Laplacian，`target_lap_vec` 是 demo/source 的目标 Laplacian。默认 `uniform_weight=True` 时，`L` 实际只由 `adj_list` 决定；`vertices` 参数只有在启用 distance-based weight 时才会影响权重。

前一节已经构造了：

```python
J_V = np.zeros((3 * V, self.nq_a))
```

`J_V` 表示 `vertices` 对 active robot variables 的 Jacobian。机器人 keypoint 行来自 MuJoCo Jacobian；物体点当前被锁定，所以物体点对应的 Jacobian 行是 0。


把 Laplacian 矩阵作用到三维坐标上，需要 Kronecker product：

```python
Kron = sp.kron(L, sp.eye(3, format="csr"), format="csr")
J_L = Kron @ J_V
```



## 3.9 约束项

`solve_single_iteration(...)` 会根据开关加入以下约束：

- Foot sticking：如果人体脚在当前帧被判定 sticking，就约束机器人对应脚 link 的 XY 位置保持在上一帧附近。输入来自 `foot_sticking_sequences[i]` 和上一帧 `q_t_last`。
- Foot lock window：如果配置了显式锁脚窗口，就在指定 frame range 内把脚 link 的 Z 约束到 `z_floor` 附近。默认不开启。
- Object / ground non-penetration：调用 `_update_jacobians_and_phis_from_q(q)` 得到候选碰撞 pair 的 signed distance `phi` 和距离 Jacobian，加入线性不穿透约束。
- Self-collision：调用 `_compute_self_collision_constraints(frame_idx)`，对配置的机器人自碰撞 body pairs 加距离约束。默认不开启。
- Joint limits：如果 `activate_joint_limits=True`，约束 `q_a_n_last + dqa` 不超过 `q_a_lb / q_a_ub`。
- Trust region：用二阶锥限制局部步长：

```python
constraints += [cp.SOC(self.step_size, dqa)]
```

不穿透约束的基本形式是：

$$
\phi(q + \Delta q) \approx \phi(q) + J_\phi \Delta q \ge -\epsilon_{pen}.
$$

整理成代码里的形式：

```python
rhs = -phi - self.penetration_tolerance
constraints += [Ja_n @ dqa >= rhs]
```

其中 `Ja_n` 是 signed distance Jacobian 在 active variables 上的切片。


## 3.10 目标函数和更新

目标函数项：

```python
obj_terms.append(cp.sum_squares(cp.multiply(sqrt_w3, lap_var - target_lap_vec)))
```

这是主要的 Laplacian matching 目标，让 current/robot mesh 的 Laplacian 接近 demo/source 的 `target_laplacian`。

其它目标项：

- Nominal tracking：augmentation 模式下，对 `track_nominal_indices` 指定的变量跟踪 `q_a_nominal`。
- `Q_diag` regularization：对某些手工指定的维度加额外二次惩罚。
- Smoothness：让当前帧更新接近上一帧到当前线性化点的差，减少时间抖动。

求解：

```python
problem = cp.Problem(cp.Minimize(cp.sum(obj_terms)), constraints)
problem.solve(solver=cp.CLARABEL, ...)
```

如果第 0 帧带 SOC trust region 求解失败，代码会移除 SOC 约束再尝试一次。

求解成功后：

```python
dqa_star = dqa.value
cost = problem.value

q_star = np.copy(q)
q_star[self.q_a_indices] = dqa_star + q_a_n_last
q_star[3:7] /= np.linalg.norm(q_star[3:7]) + 1e-12
return q_star, cost
```

- `q_star` 以当前线性化点 `q` 为背景。
- active variables 更新为 `q_a_n_last + dqa_star`。
- base quaternion 用加性更新后归一化的方式保持单位长度。
- 返回 `q_star` 和当前局部问题 cost。


## 3.11 Jacobian 计算：`_calc_manipulator_jacobians(...)`

这个函数给指定机器人 links 计算点位置和位置 Jacobian。

输入：

```python
q: np.ndarray
links: dict[str, str]
obj_frame: bool = False
point_offsets: np.ndarray | None = None
```

- `q`：当前 full qpos。
- `links`：人体 joint 名到 MuJoCo body/link 名的 mapping，例如 `"L_Wrist" -> "left_rubber_hand_link"`。
- `obj_frame`：是否把位置和 Jacobian 表达到物体坐标系。
- `point_offsets`：如果要跟踪 body 上非原点的点，则给出该点在 body frame 下的坐标；默认跟踪 body 原点。

流程：

1. 如果 `obj_frame=True`，从 `q[-7:]` 取物体 pose：`obj_pos=q[-7:-4]`，`obj_quat=q[-4:]`，构造 `obj_rot_inv=obj_rot.T`。
2. 写入 MuJoCo：

```python
self.robot_data.qpos[:] = q
mujoco.mj_forward(self.robot_model, self.robot_data)
```

3. 遍历 `links`，用 `mujoco.mj_name2id(...)` 找到 body id。
4. 调用 `_calc_contact_jacobian_from_point(body_id, pC_B)` 得到该点世界系位置 Jacobian。
5. 从 `self.robot_data.xpos[body_id]` 读取 body 原点世界位置。
6. 如果 `obj_frame=True`，做世界系到物体系变换：

```python
p_XC = obj_rot_inv @ (pos_world - obj_pos)
J_XC = obj_rot_inv @ J
```

否则：

```python
p_XC = pos_world
J_XC = J
```

输出：

```python
J_XC_dict, p_XC_dict, P_WO
```

- `p_XC_dict[name]`：第 `name` 个机器人 keypoint 在 X 坐标系下的位置。X 是 world frame 或 object frame。
- `J_XC_dict[name]`：该点位置对 active variables 的 Jacobian。
- `P_WO`：如果使用 object frame，返回物体在世界系下的 position/rotation；否则是 `None`。

注意：当前实现存储时会裁剪列：

```python
J_XC_dict[name] = J_XC[:, self.q_a_indices]
```

因此返回的 Jacobian 已经是 active variables 上的 Jacobian。


## 3.12 点 Jacobian：`_calc_contact_jacobian_from_point(...)`

这个函数返回某个 body 上点 C 的世界坐标对 full qpos 增量的 Jacobian。

输入：

- `body_idx`：MuJoCo body id。
- `p_body`：点 C 在 body frame 下的坐标；如果 `input_world=True`，则解释为世界坐标点。
- `input_world`：是否直接把 `p_body` 当作世界坐标。

流程：

1. 调用 `mujoco.mj_forward(...)` 确保 MuJoCo kinematics 是当前 qpos 下的结果。
2. 如果 `input_world=False`，用 body 位姿把局部点转到世界系：

$$
p_W = p_{WB} + R_{WB}p_B.
$$

3. 调用 MuJoCo：

```python
mujoco.mj_jac(self.robot_model, self.robot_data, Jp, Jr, p_W, int(body_idx))
```

MuJoCo 返回的 `Jp` 满足：

$$
\dot p_W = J_p^{(v)}(q)v,
$$

其中 `v` 是 MuJoCo `qvel`。

4. 调用 `_build_transform_qdot_to_qvel_fast()` 构造：

$$
v = T(q)\dot q.
$$

5. 返回：

$$
J_p^{(q)}(q) = J_p^{(v)}(q)T(q).
$$

代码：

```python
T = self._build_transform_qdot_to_qvel_fast()
return Jp @ T
```

返回值形状是 `(3, nq)`，表示：

$$
\Delta p_W \approx J_p^{(q)}\Delta q.
$$


## 3.13 `qpos` 增量到 `qvel`：`_build_transform_qdot_to_qvel_fast(...)`

MuJoCo 区分：

```text
qpos : 广义位置。free joint 用 [x, y, z, qw, qx, qy, qz]，是 7 维。
qvel : 广义速度。free joint 用 [vx, vy, vz, wx, wy, wz]，是 6 维。
```

普通 hinge / slide joint 基本是一一对应：

$$
v_i = \dot q_i.
$$

free joint 的平移部分也是单位映射：

$$
[v_x, v_y, v_z]^T = [\dot x, \dot y, \dot z]^T.
$$

free joint 的四元数导数需要转成角速度。设四元数：

$$
Q = [q_w, q_x, q_y, q_z]^T.
$$

代码使用类似：

$$
\omega = 2E(Q)\dot Q,
$$

其中：

$$
E(Q)=
\begin{bmatrix}
-q_x & q_w & q_z & -q_y \\
-q_y & -q_z & q_w & q_x \\
-q_z & q_y & -q_x & q_w
\end{bmatrix}.
$$

因此函数构造矩阵：

$$
T(q)\in\mathbb{R}^{n_v\times n_q},
\qquad v = T(q)\dot q.
$$

`_calc_contact_jacobian_from_point(...)` 用它把 MuJoCo 返回的 `qvel` Jacobian 转成优化器需要的 `qpos` 增量 Jacobian。


## 3.14 碰撞距离和 non-penetration Jacobian

object / ground non-penetration 依赖三个函数：

```text
_update_jacobians_and_phis_from_q(q)
  -> _prefilter_pairs_with_mj_collision(threshold)
  -> mujoco.mj_geomDistance(..., fromto)
  -> _compute_jacobian_for_contact_relative(...)
```

`_prefilter_pairs_with_mj_collision(threshold)`：

- 临时把 geom margin 设置为 `threshold`。
- 调用 `mujoco.mj_collision(...)` 找到候选 geom pair。
- 返回候选 pair 集合。

`_update_jacobians_and_phis_from_q(q)`：

- 输入当前线性化点 `q`。
- 对候选 pair 调用 `mujoco.mj_geomDistance(...)` 得到 signed distance `dist` 和最近点 `fromto`。
- 调用 `_compute_jacobian_for_contact_relative(...)` 计算 signed distance 对 full qpos 的 Jacobian。
- 返回两个字典：

```python
Js[(g1, g2)] = J_phi
phis[(g1, g2)] = dist
```

`_compute_jacobian_for_contact_relative(...)` 的基本公式：

$$
\phi(q) = \operatorname{sdist}(g_1(q), g_2(q)).
$$

MuJoCo 给出两个最近点 $p_1, p_2$ 后，若方向稳定：

$$
\hat n = \operatorname{sign}(\phi)\frac{p_1-p_2}{\|p_1-p_2\|},
\qquad
J_\phi = \hat n^T(J_1-J_2).
$$

其中 $J_1$、$J_2$ 由 `_calc_contact_jacobian_from_point(..., input_world=True)` 计算。

进入 CVXPY 子问题时，代码只取 active variables：

```python
Ja_n = Ja_n_full[self.q_a_indices]
constraints += [Ja_n @ dqa >= rhs]
```


## 3.15 尺度一致性 caution

当前 `object_interaction` 分支有一个需要审计的尺度事实：

1. `load_motion_data(...)` 读出原始 `human_joints/object_poses` 和 `smpl_scale`。
2. `preprocess_motion_data(...)` 会把 `human_joints` 乘以 `smpl_scale`，也会缩放 `object_poses` 的平移部分。
3. `setup_object_data(...)` 调用 `load_object_data(...)`，返回：

```python
object_local_pts      = points
object_local_pts_demo = points * smpl_scale
```

4. `retarget_motion(...)` 构造 demo/source target 时用：

```python
source_vertices = [human_mapped_joints_in_object, object_points_local_demo]
```

即“缩放后的人 + 缩放后的物体点”。

5. `solve_single_iteration(...)` 构造 current/robot 侧 mesh 时用：

```python
vertices = [robot_pts_local, obj_pts_local]
```

其中 `obj_pts_local = object_points_local = points`，即“机器人 + 原始物体点”。

因此，如果把语义理解为“机器人要和同一个原始箱子交互”，source/current 两侧物体点尺度可能不一致。这个 notebook 只记录当前实现行为，不修改源码；后续若要严格审计质量，应明确目标物体到底应该是原始尺度还是 `smpl_scale` 后的尺度，并同步检查物体 mesh、URDF/XML、`object_poses` 和 interaction mesh 点集。
